In [1]:
import base64
import getpass
from pyspark.sql import SparkSession

# 1) Capture current Spark conf (created by Watson Studio runtime)
conf = spark.sparkContext.getConf()

# 2) Stop existing Spark session
spark.stop()

# 3) Prompt for credentials (same pattern as your working notebook)
wxd_username = getpass.getpass("Please enter your watsonx.data username (HMS access): ").strip()

# In many watsonx.data setups, HMS expects username in this format:
wxd_hms_username = "ibmlhapikey_" + wxd_username

# API key used for metastore access (HMS)
wxd_hms_password = getpass.getpass("Please enter your watsonx.data API key (HMS access): ").strip()

# watsonx.data API key header format used by the Spark extension
string_to_encode = f"{wxd_username}:{wxd_hms_password}"
wxd_encoded_apikey = "ZenApiKey " + base64.b64encode(string_to_encode.encode("utf-8")).decode("utf-8")

# 4) Inject the required properties into the existing conf
conf.setAll([
    ("spark.hive.metastore.client.plain.username", wxd_hms_username),
    ("spark.hive.metastore.client.plain.password", wxd_hms_password),
    ("spark.hadoop.hive.wxd.user.name", wxd_username),
    ("spark.hadoop.wxd.apikey", wxd_encoded_apikey),

    # REQUIRED for Iceberg procedures like CALL ... rewrite_data_files
    ("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
])

# 5) Recreate Spark session using the modified conf
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()


print("spark.sql.extensions =", spark.conf.get("spark.sql.extensions", "NOT_SET"))

print("Spark created:", spark)
print("spark.sql.catalogImplementation =", spark.conf.get("spark.sql.catalogImplementation", "n/a"))
print("spark.hive.metastore.uris =", spark.conf.get("spark.hive.metastore.uris", "n/a"))

# Check if any compression is used
key = "spark.sql.parquet.compression.codec"
try:
    v = spark.conf.get(key)   # no default
except Exception:
    v = "<NOT SET>"
print(key, "=", v)

Please enter your watsonx.data username (HMS access):  ········
Please enter your watsonx.data API key (HMS access):  ········


spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Spark created: <pyspark.sql.session.SparkSession object at 0x7ff6a05975d0>
spark.sql.catalogImplementation = hive
spark.hive.metastore.uris = https://ibm-lh-lakehouse-mds-thrift-svc.cpd.svc.cluster.local:8381/mds/thrift
spark.sql.parquet.compression.codec = snappy


In [2]:
try:
    dbs = spark.catalog.listDatabases()
    print(f"✅ Metastore reachable. Databases found: {len(dbs)}")
    for d in dbs[:50]:
        print(" -", d.name)
except Exception as e:
    print("❌ Metastore NOT reachable")
    print("Exception type:", type(e).__name__)
    print(str(e))

    # Try to expose more detail if Spark attached it
    if hasattr(e, "desc"):
        print("\n--- e.desc ---")
        print(e.desc)
    if hasattr(e, "stackTrace"):
        print("\n--- e.stackTrace (first 2500 chars) ---")
        print(str(e.stackTrace)[:2500])

    raise

✅ Metastore reachable. Databases found: 5
 - default
 - wxd_system_data_diag_presto218
 - wxd_system_data_diag_presto597
 - test_hive_schema
 - wxd_system_data_diag_presto555


In [2]:
import re

conf_items = dict(spark.sparkContext.getConf().getAll())
catalog_pattern = re.compile(r"^spark\.sql\.catalog\.([^.]+)(?:\..+)?$")

catalogs = sorted({m.group(1) for k in conf_items.keys() if (m := catalog_pattern.match(k))} | {"spark_catalog"})

print("Catalogs discovered from Spark config:")
for c in catalogs:
    print(" -", c)

Catalogs discovered from Spark config:
 - hive_data
 - iceberg_data
 - sbucket01
 - spark_catalog
 - wxd_system_data


In [ ]:
MAX_SCHEMAS = 50
MAX_TABLES = 200

for s in schemas[:MAX_SCHEMAS]:
    print(f"\n== Schema: {s} ==")
    try:
        tables = spark.catalog.listTables(s)
        if not tables:
            print("  (no tables)")
            continue

        for t in tables[:MAX_TABLES]:
            print(f"  - {t.name} (type={t.tableType}, provider={getattr(t,'provider',None)})")
    except Exception as e:
        print("  ERROR:", str(e))

In [ ]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS test_hive_schema
LOCATION 's3a://hive-bucket/test_hive_schema.db'
""")

In [ ]:
from pyspark.sql import Row

data = [Row(id=1, name="Alice", age=30), Row(id=2, name="Bob", age=25)]
df = spark.createDataFrame(data)

path = "s3a://hive-bucket/test_hive_schema_data/sample_json"

df.coalesce(1).write.mode("overwrite").json(path)

df_read = spark.read.json(path)
df_read.show(truncate=False)

In [3]:
print("sbucket01 catalog class:", spark.conf.get("spark.sql.catalog.sbucket01", "<NOT SET>"))
print("sbucket01 catalog type :", spark.conf.get("spark.sql.catalog.sbucket01.type", "<NOT SET>"))
print("sbucket01 warehouse     :", spark.conf.get("spark.sql.catalog.sbucket01.warehouse", "<NOT SET>"))

# Optional: print the S3A bucket endpoint that should be used for s-bucket...
print("OCS bucket endpoint:",
      spark.conf.get("spark.hadoop.fs.s3a.bucket.s-bucket01-5b2abd42-ee8c-42f0-ac32-6f31e57990a4.endpoint", "<NOT SET>"))

sbucket01 catalog class: org.apache.iceberg.spark.SparkCatalog
sbucket01 catalog type : hive
sbucket01 warehouse     : s3a://s-bucket01-5b2abd42-ee8c-42f0-ac32-6f31e57990a4
OCS bucket endpoint: http://s3.openshift-storage.svc


In [ ]:
schema_name = "demo_schema"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS sbucket01.{schema_name}")
print("Namespace ensured:", f"sbucket01.{schema_name}")

In [ ]:
table_name = "demo_table_10rows"
full_table = f"sbucket01.{schema_name}.{table_name}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {full_table} (
  id INT,
  name STRING,
  amount DOUBLE,
  created_ts TIMESTAMP
)
USING iceberg
""")

print("Table ensured:", full_table)

In [ ]:
spark.sql(f"""
INSERT INTO {full_table} VALUES
  (1,  'row-01', 10.5,  current_timestamp()),
  (2,  'row-02', 20.0,  current_timestamp()),
  (3,  'row-03', 30.25, current_timestamp()),
  (4,  'row-04', 40.0,  current_timestamp()),
  (5,  'row-05', 50.75, current_timestamp()),
  (6,  'row-06', 60.0,  current_timestamp()),
  (7,  'row-07', 70.1,  current_timestamp()),
  (8,  'row-08', 80.0,  current_timestamp()),
  (9,  'row-09', 90.9,  current_timestamp()),
  (10, 'row-10', 100.0, current_timestamp())
""")

print("Inserted 10 rows into:", full_table)

In [4]:
schema_name = "demo_schema"
#table_name  = "big_table_100gb_t2"
table_name  = "big_table_50gb_t2"
full_table  = f"sbucket01.{schema_name}.{table_name}"

print(full_table)

sbucket01.demo_schema.big_table_50gb_t2


In [8]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS sbucket01.{schema_name}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {full_table} (
  commit_id INT,
  row_id    BIGINT,
  payload   STRING,
  ts        TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
  'format-version'='2',
  'write.delete.mode'='merge-on-read',
  'write.update.mode'='merge-on-read',
  'write.merge.mode'='merge-on-read'
)
""")

print("Table ready:", full_table)

Table ready: sbucket01.demo_schema.big_table_100gb_t2


In [5]:
from pyspark.sql import functions as F

def make_approx_gb_batch_df(commit_id: int,
                            rows_per_commit: int,
                            payload_bytes: int,
                            num_partitions: int = 200):
    """
    Creates a batch dataframe whose uncompressed raw payload size is approx:
      rows_per_commit * payload_bytes  bytes
    (not counting other columns/overhead/compression).
    """
    # Fixed-length string payload
    payload_col = F.rpad(F.lit("x"), payload_bytes, "x")

    df = (
        spark.range(0, rows_per_commit, 1, num_partitions)
             .select(
                 F.lit(commit_id).cast("int").alias("commit_id"),
                 F.col("id").cast("bigint").alias("row_id"),
                 payload_col.alias("payload"),
                 F.current_timestamp().alias("ts")
             )
    )
    return df

# Default sizing for ~1 GiB RAW payload per commit:
# 1 GiB = 1,073,741,824 bytes
# If payload is 1024 bytes, rows ~= 1,048,576 per commit
payload_bytes_default = 1024
rows_per_commit_default = 1_048_576

print("Default batch sizing:")
print(" payload_bytes =", payload_bytes_default)
print(" rows_per_commit =", rows_per_commit_default)
print(" raw_bytes_per_commit ~", rows_per_commit_default * payload_bytes_default)

Default batch sizing:
 payload_bytes = 1024
 rows_per_commit = 1048576
 raw_bytes_per_commit ~ 1073741824


In [ ]:
import time

payload_bytes   = payload_bytes_default
rows_per_commit = rows_per_commit_default
#num_commits     = 100
#num_commits     = 20
num_commits     = 50

# If you find it slow/heavy, increase partitions or reduce payload/rows.
num_partitions  = 100
#num_partitions  = 200

start = time.time()

for commit_id in range(1, num_commits + 1):
    batch_df = make_approx_gb_batch_df(
        commit_id=commit_id,
        rows_per_commit=rows_per_commit,
        payload_bytes=payload_bytes,
        num_partitions=num_partitions
    )

    # Append to Iceberg table (each append ~ one commit)
    # Using saveAsTable is widely compatible with Iceberg integrations.
    batch_df.write.format("iceberg").mode("append").saveAsTable(full_table)

    elapsed = time.time() - start
    approx_gb = (commit_id * rows_per_commit * payload_bytes) / (1024**3)
    print(f"Committed batch {commit_id}/{num_commits} | approx raw written: {approx_gb:.1f} GiB | elapsed: {elapsed/60:.1f} min")

print("DONE.")

Committed batch 1/50 | approx raw written: 1.0 GiB | elapsed: 1.1 min
Committed batch 2/50 | approx raw written: 2.0 GiB | elapsed: 1.7 min
Committed batch 3/50 | approx raw written: 3.0 GiB | elapsed: 2.2 min
Committed batch 4/50 | approx raw written: 4.0 GiB | elapsed: 2.9 min


In [ ]:
expected = num_commits * rows_per_commit
actual = spark.table(full_table).count()

print("Expected rows:", expected)
print("Actual rows  :", actual)

In [ ]:
spark.sql(f"""
SELECT commit_id, COUNT(*) AS rows
FROM {full_table}
GROUP BY commit_id
ORDER BY commit_id
""").show(120, truncate=False)

In [9]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [9]:
spark.conf.get("spark.sql.catalog.sbucket01")

'org.apache.iceberg.spark.SparkCatalog'

In [6]:
spark.sql("SELECT version()").show()

+--------------------+
|           version()|
+--------------------+
|3.5.4 a6f220d9517...|
+--------------------+



In [5]:
spark.conf.get("spark.sql.extensions", "NOT_SET")

'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'

In [ ]:
spark.sql("""
CALL sbucket01.system.rewrite_data_files('demo_schema.big_table_100gb_t2')
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|20000                     |1                     |300093850            |0                      |0                         |
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [4]:
spark.sql("""
CALL sbucket01.system.rewrite_manifests('demo_schema.big_table_100gb_t2')
""").show(truncate=False)

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|51                       |1                    |
+-------------------------+---------------------+



In [42]:
spark.sql("select * from sbucket01.demo_schema.big_table_100gb_mor.snapshots").count()

102

In [15]:
spark.sql("select * from sbucket01.demo_schema.big_table_100gb_mor.snapshots").show(1000,truncate=False)

+-----------------------+-------------------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                               

In [3]:
spark.sql("""
CALL sbucket01.system.set_current_snapshot('demo_schema.big_table_100gb_mor',7858163571728226348)
""").show(truncate=False)

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|3035340511197955123 |7858163571728226348|
+--------------------+-------------------+



In [16]:
spark.sql("select * from sbucket01.demo_schema.big_table_100gb_mor.manifests VERSION AS OF 7858163571728226348").show(10000,truncate=False)

+-------+----------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path                                                                                                                                          |length|partition_spec_id|added_snapshot_id  |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+----------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+----------

In [ ]:
spark.sql("select * from sbucket01.demo_schema.big_table_100gb_mor where row_id=100").show(1000,truncate=False)

In [ ]:
spark.sql("desc extended sbucket01.demo_schema.big_table_100gb_mor").show(1000,truncate=False)

In [ ]:
p